In [41]:
!pip install fastapi uvicorn python-multipart scikit-learn joblib

In [42]:
import os

print(os.listdir("/kaggle/input/datasets/vaclavpechtor/rvl-cdip-small-200/rvl-cdip-small-200"))

['val', 'train']


In [43]:
print(os.listdir("/kaggle/input/datasets/vaclavpechtor/rvl-cdip-small-200/rvl-cdip-small-200/train"))

['scientific_report', 'resume', 'memo', 'file_folder', 'specification', 'news_article', 'letter', 'form', 'budget', 'handwritten', 'email', 'invoice', 'presentation', 'scientific_publication', 'questionnaire', 'advertisement']


In [45]:
import os
from PIL import Image
import pytesseract

base_dir = "/kaggle/input/datasets/vaclavpechtor/rvl-cdip-small-200/rvl-cdip-small-200"

def load_documents(base_dir, selected_classes):

    documents = []
    labels = []

    for split in ["train", "val"]:
        split_path = os.path.join(base_dir, split)

        print(f"\n📂 Processing: {split}")

        for class_name in os.listdir(split_path):

            if class_name not in selected_classes:
                continue

            class_path = os.path.join(split_path, class_name)

            print("➡ Using class:", class_name)

            for file in os.listdir(class_path):
                file_path = os.path.join(class_path, file)

                try:
                    img = Image.open(file_path)
                    text = pytesseract.image_to_string(img)

                    if text.strip():   # ignore empty OCR
                        documents.append(text)
                        labels.append(class_name)

                except Exception as e:
                    print("Error:", file_path)

    return documents, labels

In [46]:
selected_classes = ["invoice", "letter", "resume"]

documents, labels = load_documents(base_dir, selected_classes)

print("\nTotal docs:", len(documents))
print("Classes:", set(labels))


📂 Processing: train
➡ Using class: resume
➡ Using class: letter
➡ Using class: invoice

📂 Processing: val
➡ Using class: resume
➡ Using class: letter
➡ Using class: invoice

Total docs: 596
Classes: {'invoice', 'resume', 'letter'}


In [47]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    documents,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

# Create TF-IDF vectorizer
vectorizer = TfidfVectorizer(
    max_features=1000,
    stop_words='english',
    ngram_range=(1, 2)   # unigrams + bigrams
)

# Fit + transform
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# Train classifier
classifier = LogisticRegression(max_iter=1000)
classifier.fit(X_train_vec, y_train)

# Predict
y_pred = classifier.predict(X_test_vec)

# Evaluate
accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.2%}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred)) 

Accuracy: 96.67%

Classification Report:
              precision    recall  f1-score   support

     invoice       0.97      0.93      0.95        40
      letter       0.93      0.97      0.95        40
      resume       1.00      1.00      1.00        40

    accuracy                           0.97       120
   macro avg       0.97      0.97      0.97       120
weighted avg       0.97      0.97      0.97       120



In [48]:
import joblib
import os

os.makedirs("models", exist_ok=True)

joblib.dump(vectorizer, "models/vectorizer.pkl")
joblib.dump(classifier, "models/classifier.pkl")

print("Models saved successfully!")

loaded_vec = joblib.load("models/vectorizer.pkl")
loaded_clf = joblib.load("models/classifier.pkl")

print("Models loaded successfully!")

Models saved successfully!
Models loaded successfully!
